In [5]:
import os
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI

# 1. Load API Key
load_dotenv(dotenv_path=".env")
api_key = os.getenv("GOOGLE_API_KEY")

# 2. Initialize Gemini 2.5 Flash
# Note: In 2026, 'gemini-2.5-flash' is the standard production alias.
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash", 
    google_api_key=api_key,
    temperature=0
)

# 3. Setup Logic
prompt = PromptTemplate.from_template("Translate to French: {text}")
parser = StrOutputParser()

# 4. Chain & Invoke
chain = prompt | llm | parser

try:
    response = chain.invoke({"text": "Good morning, how can I help you today?"})
    print(f"Gemini 2.5 Response: {response}")
except Exception as e:
    print(f"Error: {e}")

Gemini 2.5 Response: **Bonjour, comment puis-je vous aider aujourd'hui ?**

You could also say:
*   **Bonjour, que puis-je faire pour vous aujourd'hui ?** (Good morning, what can I do for you today?)


In [6]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import CommaSeparatedListOutputParser

# 1. Load API Key
load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")

# 2. Initialize the Parser and get instructions
parser = CommaSeparatedListOutputParser()
format_instructions = parser.get_format_instructions()

# 3. Setup Prompt with format instructions
# We use 'partial_variables' to inject the parser's specific instructions
prompt = PromptTemplate(
    template="List 5 {subject}.\n{format_instructions}",
    input_variables=["subject"],
    partial_variables={"format_instructions": format_instructions}
)

# 4. Initialize Gemini
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

# 5. Build and Invoke Chain
chain = prompt | llm | parser

response = chain.invoke({"subject": "programming languages"})
print(response) 
# Output: ['Python', 'Java', 'C++', 'JavaScript', 'Ruby']

['Python', 'Java', 'C++', 'JavaScript', 'Ruby']


In [8]:
import os
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

# 1. Define your Schema
class ProductInfo(BaseModel):
    name: str = Field(description="Name of the product")
    price: float = Field(description="Price in INR")

# 2. Setup Parser and Model
parser = PydanticOutputParser(pydantic_object=ProductInfo)
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

# 3. Setup Prompt with format instructions
prompt = PromptTemplate(
    template="Extract product name and price from: {text}\n{format_instructions}",
    input_variables=["text"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

# 4. Chain and Invoke
chain = prompt | llm | parser

response = chain.invoke({
    "text": "The Redmi Note 12 is available for ₹14,999."
})

print(response) 
# Output: name='Redmi Note 12' price=14999.0

name='Redmi Note 12' price=14999.0
